# Simplified EEG–BI Respiration Check

Purpose: first answer one basic question:

**Does low-frequency EEG activity from Fp1/Fp2 line up with BI-derived breathing?**

This notebook intentionally leaves out PAC, bycycle, advanced ICA interpretation, and extra metrics until the time alignment is clear.

In [ ]:
# Cell 1 — Imports

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

from scipy.signal import butter, filtfilt, welch, correlate
from scipy.interpolate import interp1d
from scipy.stats import pearsonr
from scipy.ndimage import gaussian_filter1d

In [ ]:
# Cell 2 — File paths

eeg_set_path = "/Users/sarah-rosemay/Documents/EEG_data/EEG_DATA_05_4_2026/Halo_4ch_ica.set"

# EmotiBit CSV with HR / BI rows
ppg_csv_path = "/Users/sarah-rosemay/Documents/EEG_data/Combo test/2026-04-27_11-49-02-823686.csv"

In [ ]:
# Cell 3 — Helper functions

def zscore(x):
    x = np.asarray(x, dtype=float)
    return (x - np.nanmean(x)) / np.nanstd(x)


def bandpass_filter(x, fs, low=0.1, high=0.4, order=4):
    nyq = fs / 2
    b, a = butter(order, [low / nyq, high / nyq], btype="band")
    return filtfilt(b, a, x)


def dominant_freq(x, fs, fmin=0.05, fmax=0.5):
    freqs, psd = welch(x, fs=fs, nperseg=min(4096, len(x)))
    keep = (freqs >= fmin) & (freqs <= fmax)
    peak_f = freqs[keep][np.argmax(psd[keep])]
    return peak_f, peak_f * 60, freqs, psd


def best_lag_seconds(x, y, fs, max_lag_sec=120):
    # x and y must have same length and sampling rate
    xz = zscore(x)
    yz = zscore(y)
    max_lag = int(max_lag_sec * fs)

    corr = correlate(xz, yz, mode="full")
    lags = np.arange(-len(xz) + 1, len(xz))

    keep = np.abs(lags) <= max_lag
    corr = corr[keep]
    lags = lags[keep]

    best_i = np.argmax(np.abs(corr))
    best_lag = lags[best_i] / fs

    return best_lag, corr[best_i]


def shift_signal_by_seconds(y, fs, lag_sec):
    # Positive lag means y is shifted forward relative to x.
    shift = int(round(lag_sec * fs))
    y_shifted = np.roll(y, shift)

    if shift > 0:
        y_shifted[:shift] = np.nan
    elif shift < 0:
        y_shifted[shift:] = np.nan

    return y_shifted

In [ ]:
# Cell 4 — Load EEG

raw = mne.io.read_raw_eeglab(eeg_set_path, preload=True)

print(raw)
print("Channels:", raw.ch_names)
print("Sampling rate:", raw.info["sfreq"])

fs_eeg = raw.info["sfreq"]

# Keep only frontal channels for the first simple check
raw_fp = raw.copy().pick(["Fp1", "Fp2"])

# Get data in whatever unit MNE loaded from the EEGLAB file
fp_data = raw_fp.get_data()
times_eeg = raw_fp.times

print("Fp data shape:", fp_data.shape)
print("EEG duration sec:", times_eeg[-1])

In [ ]:
# Cell 5 — Filter Fp1/Fp2 into breathing band

fp1 = fp_data[0]
fp2 = fp_data[1]

fp1_breath = bandpass_filter(fp1, fs_eeg, low=0.1, high=0.4)
fp2_breath = bandpass_filter(fp2, fs_eeg, low=0.1, high=0.4)

plt.figure(figsize=(14, 4))
plt.plot(times_eeg, zscore(fp1_breath), label="Fp1 0.1–0.4 Hz")
plt.plot(times_eeg, zscore(fp2_breath), label="Fp2 0.1–0.4 Hz", alpha=0.7)
plt.xlabel("EEG time (sec)")
plt.ylabel("Z-scored amplitude")
plt.title("EEG breathing-band signals")
plt.legend()
plt.show()

for name, sig in [("Fp1", fp1_breath), ("Fp2", fp2_breath)]:
    peak_hz, bpm, freqs, psd = dominant_freq(sig, fs_eeg)
    print(f"{name} peak: {peak_hz:.3f} Hz = {bpm:.1f} breaths/min")

In [ ]:
# Cell 6 — Load BI from EmotiBit CSV

ppg = pd.read_csv(
    ppg_csv_path,
    engine="python",
    header=None,
    on_bad_lines="skip"
)

print("CSV shape:", ppg.shape)
print("Detected labels:")
print(ppg[3].value_counts().head(20))

bi = ppg[ppg[3] == "BI"].copy()
print("BI rows:", bi.shape)

# In your previous file, BI values were in column 6.
bi_vals = pd.to_numeric(bi[6], errors="coerce").dropna().values

print("First 10 BI values:", bi_vals[:10])
print("BI min/max:", np.nanmin(bi_vals), np.nanmax(bi_vals))

In [ ]:
# Cell 7 — Convert BI to a simple breathing proxy

# BI is irregular because each value occurs per heartbeat, not at a fixed EEG sampling rate.
# For this first clean check, treat BI index as its own time-like axis.
# Later, if EmotiBit timestamps are available and reliable, replace this with true timestamps.

bi_smooth = gaussian_filter1d(bi_vals.astype(float), sigma=3)

# Make a normalized BI breathing proxy.
# Depending on physiology and preprocessing, this may be inverted relative to EEG.
bi_proxy = zscore(bi_smooth)

plt.figure(figsize=(14, 4))
plt.plot(bi_proxy)
plt.xlabel("BI sample index")
plt.ylabel("Z-scored BI")
plt.title("BI-derived respiration proxy")
plt.show()

In [ ]:
# Cell 8 — Put BI proxy onto the EEG time grid

# Since BI does not have reliable timestamps here, we stretch the BI series across the EEG duration.
# This is NOT final synchronization. It is only a first visual/debugging check.

bi_time_stretched = np.linspace(0, times_eeg[-1], len(bi_proxy))

interp_bi = interp1d(
    bi_time_stretched,
    bi_proxy,
    kind="linear",
    bounds_error=False,
    fill_value=np.nan
)

bi_on_eeg_time = interp_bi(times_eeg)

plt.figure(figsize=(14, 4))
plt.plot(times_eeg, bi_on_eeg_time, label="BI proxy stretched to EEG duration")
plt.xlabel("Time (sec)")
plt.ylabel("Z-scored BI")
plt.title("BI proxy on EEG time grid")
plt.legend()
plt.show()

In [ ]:
# Cell 9 — Compare EEG breathing-band signal to BI before ICA

# Use the average of Fp1 and Fp2 as the simplest frontal respiration candidate.
eeg_resp_simple = zscore((fp1_breath + fp2_breath) / 2)

valid = np.isfinite(eeg_resp_simple) & np.isfinite(bi_on_eeg_time)

r, p = pearsonr(eeg_resp_simple[valid], bi_on_eeg_time[valid])

print(f"Zero-lag correlation: r = {r:.3f}, p = {p:.3g}")

plt.figure(figsize=(14, 5))
plt.plot(times_eeg, eeg_resp_simple, label="EEG Fp1/Fp2 avg, 0.1–0.4 Hz")
plt.plot(times_eeg, bi_on_eeg_time, label="BI proxy, stretched", alpha=0.8)
plt.xlabel("Time (sec)")
plt.ylabel("Z-scored amplitude")
plt.title("First-pass EEG vs BI comparison BEFORE ICA")
plt.legend()
plt.show()

In [ ]:
# Cell 10 — Lag scan: does shifting BI improve the match?

valid = np.isfinite(eeg_resp_simple) & np.isfinite(bi_on_eeg_time)

x = eeg_resp_simple[valid]
y = bi_on_eeg_time[valid]

lag_sec, lag_score = best_lag_seconds(x, y, fs_eeg, max_lag_sec=120)

print(f"Best lag within ±120 sec: {lag_sec:.2f} sec")
print("Positive lag means BI was shifted forward relative to EEG.")

bi_shifted = shift_signal_by_seconds(bi_on_eeg_time, fs_eeg, lag_sec)

valid2 = np.isfinite(eeg_resp_simple) & np.isfinite(bi_shifted)
r_shifted, p_shifted = pearsonr(eeg_resp_simple[valid2], bi_shifted[valid2])

print(f"After lag correction: r = {r_shifted:.3f}, p = {p_shifted:.3g}")

plt.figure(figsize=(14, 5))
plt.plot(times_eeg, eeg_resp_simple, label="EEG Fp1/Fp2 avg, 0.1–0.4 Hz")
plt.plot(times_eeg, bi_shifted, label=f"BI proxy shifted by {lag_sec:.1f} sec", alpha=0.8)
plt.xlabel("Time (sec)")
plt.ylabel("Z-scored amplitude")
plt.title("EEG vs BI after simple lag correction")
plt.legend()
plt.show()

In [ ]:
# Cell 11 — Only now: optional ICA comparison

# This uses the EEGLAB ICA weights already saved in the .set file.
# With only Fp1/Fp2, ICA components are just weighted mixtures of Fp1 and Fp2.
# Do not over-interpret them as clean brain sources.

try:
    eeglab = raw._raw_extras[0]

    icaweights = np.array(eeglab["icaweights"], dtype=float)
    icasphere = np.array(eeglab["icasphere"], dtype=float)

    data = raw.get_data()
    unmixing = icaweights @ icasphere
    icaact_reconstructed = unmixing @ data

    print("ICA activity shape:", icaact_reconstructed.shape)

    ica_breath = np.array([
        bandpass_filter(comp, fs_eeg, low=0.1, high=0.4)
        for comp in icaact_reconstructed
    ])

    rows = []
    for i, comp in enumerate(ica_breath):
        comp_z = zscore(comp)
        valid = np.isfinite(comp_z) & np.isfinite(bi_shifted)
        r, p = pearsonr(comp_z[valid], bi_shifted[valid])
        peak_hz, bpm, _, _ = dominant_freq(comp_z[valid], fs_eeg)

        rows.append({
            "signal": f"ICA {i}",
            "corr_with_shifted_BI": r,
            "p_value": p,
            "peak_freq_Hz": peak_hz,
            "peak_breaths_per_min": bpm
        })

    results = pd.DataFrame(rows)
    display(results)

except Exception as e:
    print("ICA section skipped because ICA info was not available or could not be reconstructed.")
    print(type(e).__name__, e)

## Interpretation checklist

Use this before adding advanced analyses back:

1. Do Fp1/Fp2 and BI have similar dominant breathing frequencies?
2. Do the waveforms visually line up?
3. Does correlation improve after a lag shift?
4. Is ICA meaningfully better than Fp1/Fp2 average?
5. Only after these are clear should you add PAC, bycycle, or cycle-by-cycle metrics.